<a href="https://colab.research.google.com/github/mohsinam-afk/Sprint-1---Microscopic-cell-counting-/blob/main/Base_Model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WELCOME TO MY SPRINT 1 WORKBOOK: This the Base Model  

This sections loads the requied libraries and the datset is downloaded in the cloud from the URL of the dataset directly

In [1]:
import os
import glob
import cv2
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
import re

# 1. Download the dataset from the real direct data URL
print("Downloading the BBBC005 1.8GB archive from the Broad Institute server...")
# Note: The data file sits on the data subdomain, not the main bbbc site
DATA_URL = "https://data.broadinstitute.org/bbbc/BBBC005/BBBC005_v1_images.zip"
!wget -O /content/BBBC005_v1_images.zip "{DATA_URL}"

# 2. Extract the downloaded dataset into temorary cloud memory
print("Extracting images into cloud memory...")
!unzip -q /content/BBBC005_v1_images.zip -d /content/dataset
print("Extraction complete!")

--2026-07-01 09:46:15--  https://data.broadinstitute.org/bbbc/BBBC005/BBBC005_v1_images.zip
Resolving data.broadinstitute.org (data.broadinstitute.org)... 69.173.68.137
Connecting to data.broadinstitute.org (data.broadinstitute.org)|69.173.68.137|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1882973059 (1.8G) [application/zip]
Saving to: ‘/content/BBBC005_v1_images.zip’

/content/BBBC005_v1 100%[===================>]   1.75G  35.3MB/s    in 48s     

2026-07-01 09:47:03 (37.7 MB/s) - ‘/content/BBBC005_v1_images.zip’ saved [1882973059/1882973059]

Extracting images into cloud memory...
Extraction complete!


In [ ]:
# 3. Verify the images are stored on corect folders  and a patch of images are used to test the logic
#There are 19200 images in the dataset starting from index 0 to 19199
dataset_dir = "/content/dataset/BBBC005_v1_images"


In [ ]:
# ======================================================================================
# Initialize the storage counters/placeholders
# ======================================================================================
total_images = 0      # Track how many image files are successfully read
total_cells = 0       # Cumulative counter for all cells detected across the entire batch
sample_plotted = False # Controls generating the visual breakout plot exactly once

image_paths = sorted(glob.glob(os.path.join(dataset_dir, "*.TIF")))
print("Total images found:", len(image_paths))

In [ ]:
# Loop through every file inside the directory
for filename in os.listdir(dataset_dir):
    img_path = os.path.join(dataset_dir, filename)

    img = cv2.imread(img_path, 0)

    # FIX: Skip non-image files or failed reads
    if img is None:
        continue

    total_images += 1

    # Create a structuring element
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # --------------------------------------------------
    # Step 1: Otsu Thresholding View
    # --------------------------------------------------
    thresh2, bw2 = cv2.threshold(img, thresh=127, maxval=255, type=cv2.THRESH_OTSU)

    # --------------------------------------------------
    # Step 2: Morphological Opening (Erosion followed by Dilation)
    # --------------------------------------------------
    opening1 = cv2.morphologyEx(bw2, cv2.MORPH_OPEN, kernel, iterations=1)

    # --------------------------------------------------
    # Step 3 & 4: Continuous Erosion Loop
    # --------------------------------------------------
    erosion = cv2.erode(opening1, kernel, iterations=1)
    for i in range(1, 4):
        erosion = cv2.erode(erosion, kernel, iterations=1)

    # --------------------------------------------------
    # Step 5: Labels & Centroid Marker Drawing
    # --------------------------------------------------
    labels, nlabels = ndimage.label(erosion)
    total_cells += nlabels  # Accumulate data across the entire dataset

    centroid = ndimage.center_of_mass(erosion, labels, np.arange(nlabels) + 1)

    display = img.copy()
    display = cv2.cvtColor(display, cv2.COLOR_GRAY2BGR)

    for pt in centroid:
        if not np.isnan(pt).any():
            # FIXED Indexing: ndimage (row, col) maps to cv2 (x, y) as (col, row)
            center = (int(pt[1]), int(pt[0]))
            cv2.circle(display, center, radius=3, color=(0, 0, 255), thickness=-1)

    # --------------------------------------------------
    # Plot Only One Image Output with All Intermediate Steps
    # --------------------------------------------------
    if not sample_plotted:
        plt.figure(figsize=(18, 10))

        # 1. Original Grayscale Image
        plt.subplot(1, 5, 1)
        plt.imshow(img, cmap="gray")
        plt.title("1. Original Image")
        plt.axis("off")

        # 2. After Thresholding (Otsu)
        plt.subplot(1, 5, 2)
        plt.imshow(bw2, cmap="gray")
        plt.title("2. After Threshold")
        plt.axis("off")

        # 3. Intermediate state after Erosion (Final erosion layer before tracking)
        plt.subplot(1, 5, 3)
        plt.imshow(erosion, cmap="gray")
        plt.title("3. After Erosion")
        plt.axis("off")

        # 4. Morphological Opening (Which contains the dilation step of opening)
        plt.subplot(1, 5, 4)
        plt.imshow(opening1, cmap="gray")
        plt.title("4. After Dilation (Opening)")
        plt.axis("off")

        # 5. Final Image with Marked Red Nuclei
        plt.subplot(1, 5, 5)
        plt.imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
        plt.title(f"5. Red Nuclei ({nlabels} Cells)")
        plt.axis("off")

        plt.suptitle(f"Processing Pipeline Breakdown for Sample: {filename}", fontsize=16, y=0.85)
        plt.tight_layout()
        plt.show()

        # Extract Ground Truth Count and Print Sample Comparison right away
        ground_truth_cells = int(re.search(r'_C(\d+)_', filename).group(1))
        print("\n" + "="*40)
        print(f" SINGLE IMAGE VALIDATION METRICS")
        print("="*40)
        print(f" Filename Label     : {filename}")
        print(f" Ground Truth Cells : {ground_truth_cells}")
        print(f" Model Found Cells  : {nlabels}")
        print(f" Counting Deviation : {abs(nlabels - ground_truth_cells)} cells")
        print("="*40 + "\n")

        sample_plotted = True

print("Dataset iteration completed successfully.")


In [29]:
# --------------------------------------------------
# Final Dataset-Wide Output Metric
# --------------------------------------------------
print("=" * 50)
print(f"Dataset Processing Summary:")
print("=" * 50)
print(f"Total number of images: {total_images}")
print(f"Total number of cells counted: {total_cells}")
print("=" * 50)


Dataset Processing Summary:
Total number of images: 19200
Total number of cells counted: 506262
